# 3. Train / Validation / Test Strategy

This notebook defines the data splitting strategy for the machine learning pipeline.

The objective is to evaluate the model in a way that approximates real-world future delivery predictions while avoiding data leakage.

Because the dataset contains repeated delivery personnel and observations across multiple dates, random splitting may produce an overly optimistic estimate of generalization.

This phase will:

1. Inspect the temporal coverage of the dataset.
2. Inspect repeated delivery personnel.
3. Compare possible splitting strategies.
4. Select a final train/validation/test strategy.
5. Freeze the resulting split before model training.

In [24]:
# Load the prepared Datset
from pathlib import Path
import numpy as np
import pandas as pd

In [25]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

print("Project root :", PROJECT_ROOT)
print("Processed data path :", PROCESSED_DATA_PATH)

Project root : c:\Users\A MAHESH\OneDrive\Desktop\INNOMATICS-DS\ML project\Last-mile-Delivery-Intillegence
Processed data path : c:\Users\A MAHESH\OneDrive\Desktop\INNOMATICS-DS\ML project\Last-mile-Delivery-Intillegence\data\processed


In [26]:
list(PROCESSED_DATA_PATH.iterdir())

[WindowsPath('c:/Users/A MAHESH/OneDrive/Desktop/INNOMATICS-DS/ML project/Last-mile-Delivery-Intillegence/data/processed/cleaned_data.csv')]

In [27]:
df_clean = pd.read_csv(PROCESSED_DATA_PATH / 'cleaned_data.csv')

In [28]:
print('Shape :',df_clean.shape)
print('\nDate range :')
print("Start :",df_clean['order_date'].min())
print("End:  ",df_clean['order_date'].max())

Shape : (41953, 38)

Date range :
Start : 2022-02-11
End:   2022-04-06


In [29]:
df_clean['order_date'].describe()

count          41953
unique            44
top       2022-03-15
freq            1115
Name: order_date, dtype: object

In [30]:
date_distribution = (df_clean['order_date'].value_counts().sort_index())

date_distribution.head(10)

order_date
2022-02-11     799
2022-02-12     717
2022-02-13     785
2022-02-14     707
2022-02-15     785
2022-02-16     727
2022-02-17     776
2022-02-18     720
2022-03-01    1071
2022-03-02     944
Name: count, dtype: int64

In [31]:
date_distribution.tail(10)

order_date
2022-03-28    1065
2022-03-29     914
2022-03-30    1068
2022-03-31     904
2022-04-01    1060
2022-04-02     933
2022-04-03    1101
2022-04-04     886
2022-04-05    1091
2022-04-06     894
Name: count, dtype: int64

In [32]:
# Check Driver repetition
driver_counts = (df_clean["Delivery_person_ID"].value_counts() )
driver_counts.describe() 

count    1170.000000
mean       35.857265
std        21.131329
min         5.000000
25%        13.000000
50%        43.000000
75%        56.000000
max        67.000000
Name: count, dtype: float64

In [33]:
print('Unique delivery personnel :',df_clean['Delivery_person_ID'].nunique())

print('Total_observations :',len(df_clean))

print('Drivers appearing more than once :',(driver_counts > 1).sum())

Unique delivery personnel : 1170
Total_observations : 41953
Drivers appearing more than once : 1170


In [34]:
driver_counts.head(10)

Delivery_person_ID
JAPRES11DEL02     67
PUNERES01DEL01    67
VADRES11DEL02     66
JAPRES03DEL01     66
HYDRES04DEL02     66
SURRES11DEL01     65
INDORES08DEL02    65
VADRES11DEL01     65
VADRES14DEL01     65
BANGRES07DEL02    65
Name: count, dtype: int64

In [35]:
df_clean['order_date'] = pd.to_datetime(
    df_clean['order_date'],errors = 'coerce'
)
print("Data type :",df_clean['order_date'].dtype)
print('Missing_dates :',df_clean['order_date'].isna().sum())

Data type : datetime64[us]
Missing_dates : 0


In [36]:
# Get Unique Dates

unique_dates = (df_clean['order_date']
                .dropna()
                .sort_values()
                .unique() )

print("Number of unique dates :",len(unique_dates))
print('\nFirst 10 dates :')
print(unique_dates[:10])
print("\nLast 10 dates:")
print(unique_dates[10:])

Number of unique dates : 44

First 10 dates :
<DatetimeArray>
['2022-02-11 00:00:00', '2022-02-12 00:00:00', '2022-02-13 00:00:00',
 '2022-02-14 00:00:00', '2022-02-15 00:00:00', '2022-02-16 00:00:00',
 '2022-02-17 00:00:00', '2022-02-18 00:00:00', '2022-03-01 00:00:00',
 '2022-03-02 00:00:00']
Length: 10, dtype: datetime64[us]

Last 10 dates:
<DatetimeArray>
['2022-03-03 00:00:00', '2022-03-04 00:00:00', '2022-03-05 00:00:00',
 '2022-03-06 00:00:00', '2022-03-07 00:00:00', '2022-03-08 00:00:00',
 '2022-03-09 00:00:00', '2022-03-10 00:00:00', '2022-03-11 00:00:00',
 '2022-03-12 00:00:00', '2022-03-13 00:00:00', '2022-03-14 00:00:00',
 '2022-03-15 00:00:00', '2022-03-16 00:00:00', '2022-03-17 00:00:00',
 '2022-03-18 00:00:00', '2022-03-19 00:00:00', '2022-03-20 00:00:00',
 '2022-03-21 00:00:00', '2022-03-23 00:00:00', '2022-03-24 00:00:00',
 '2022-03-25 00:00:00', '2022-03-26 00:00:00', '2022-03-27 00:00:00',
 '2022-03-28 00:00:00', '2022-03-29 00:00:00', '2022-03-30 00:00:00',
 '2022-0

In [37]:
n_dates = len(unique_dates)

train_end_idx = int(n_dates * 0.70)
validation_end_idx = int(n_dates * 0.85)

train_end_date = unique_dates[train_end_idx - 1]
validation_end_date = unique_dates[validation_end_idx - 1]

print("Train ends : ",train_end_date)
print("Validation ends :",validation_end_date)
print('Test ends :',unique_dates[-1])

Train ends :  2022-03-23 00:00:00
Validation ends : 2022-03-30 00:00:00
Test ends : 2022-04-06 00:00:00


In [38]:
# Splitting in to train, validation and split
train_df = df_clean[df_clean['order_date'] <= train_end_date].copy()

validation_df = df_clean[(df_clean['order_date'] > train_end_date) & 
                         (df_clean['order_date'] <= validation_end_date)]


test_df = df_clean[df_clean['order_date'] > validation_end_date].copy()

In [39]:
print('Train shape :',train_df.shape)
print('Validation shape :',validation_df.shape)
print("Test Shape :",test_df.shape)

Train shape : (28034, 38)
Validation shape : (7050, 38)
Test Shape : (6869, 38)


In [40]:
# checking Date Range

print('\nTrain date range :')
print(train_df['order_date'].min(), '->', train_df['order_date'].max())

print('\nValidation date range :')
print(validation_df['order_date'].min() ,'->',validation_df['order_date'].max())

print('\nTest date range :')
print(test_df['order_date'].min(), '->',test_df['order_date'].max())



Train date range :
2022-02-11 00:00:00 -> 2022-03-23 00:00:00

Validation date range :
2022-03-24 00:00:00 -> 2022-03-30 00:00:00

Test date range :
2022-03-31 00:00:00 -> 2022-04-06 00:00:00


In [41]:
# checking of both train,test,validation rows are same as Actual Cleaned Data

print("\nTotal rows :",
      len(train_df) + len(validation_df) + len(test_df)) 

print("\nOriginal Rows :",
      len(df_clean))


Total rows : 41953

Original Rows : 41953


In [42]:
train_drivers = set(train_df['Delivery_person_ID'])
validation_drivers = set(validation_df['Delivery_person_ID'])
test_drivers = set(test_df['Delivery_person_ID'])

print("Train drivers:", len(train_drivers))
print("Validation drivers:", len(validation_drivers))
print("Test drivers:", len(test_drivers))


print("\nTrain ∩ Validation:",
      len(train_drivers & validation_drivers))

print("Train ∩ Test:",
      len(train_drivers & test_drivers))

print("Validation ∩ Test:",
      len(validation_drivers & test_drivers))

Train drivers: 1170
Validation drivers: 675
Test drivers: 675

Train ∩ Validation: 675
Train ∩ Test: 675
Validation ∩ Test: 675


## Final Split Strategy

A chronological train/validation/test split is used to simulate future delivery-time prediction.

- Training set: earliest observations
- Validation set: subsequent observations used for model selection and tuning
- Test set: latest observations reserved for final evaluation

The split is based on unique calendar dates rather than row counts.

Delivery personnel may appear across multiple splits. This reflects the intended production scenario where the system may predict future deliveries for personnel with historical observations.

The test set is therefore interpreted as a **future-time holdout**, not an unseen-driver generalization test.